In [1]:
import numpy as np
import pandas as pd

## Getting Baseline Data

In [3]:
df_baseline = pd.read_csv("../docs/zone_baseline_matrix.csv")

## Creating Synthetic Dataset
We join our baseline data with a datetime set that spans for a month with one hour increments, on each created row we add a "slowdown" amount based on preexisting rules

In [5]:
SEED = 42
START = "2026-10-05 00:00"   
NOISE = .04
WEEKS = 4
FIRST_HOUR_SEC = 3600       
rng = np.random.default_rng(SEED)

assert pd.Timestamp(START).dayofweek == 0, "START must be a Monday"

# columns: origin_id, destination_id, baseline_seconds, baseline_meters
base = df_baseline.dropna(subset=["baseline_seconds"])[
    ["origin_id", "destination_id", "baseline_seconds", "baseline_meters"]
].copy()
base.rename(columns={"origin_id": "origin_zone_id", "destination_id": "destination_zone_id"},inplace= True)

times = pd.DataFrame(
    {"started_at": pd.date_range(START, periods=24 * 7 * WEEKS, freq="h")}
)
times["date"] = times["started_at"].dt.normalize()
dow = times["started_at"].dt.dayofweek.to_numpy()   # Mon=0 ... Sun=6
hour = times["started_at"].dt.hour.to_numpy()

def day_range(d):
    if d == 3:           # Thu
        return (1.0, 1.5)
    if d in (4, 5):      # Fri, Sat
        return (1.5, 2.0)
    return (1, 1.1)    # Mon, Tue, Wed, Sun

dates = pd.DatetimeIndex(times["date"].unique())

# One slowdown value per day of the week
day_draw = {
    dt: rng.uniform(*day_range(dt.dayofweek)) for dt in dates
}
times["day_mult"] = times["date"].map(day_draw)

# One slowdown value per hour
hour_edges = np.array([0, 5, 8, 11, 15, 16, 18, 20, 24])
hour_lo    = np.array([1.0, 1.0, 0.9, 1.3, 1.0, 1.5, 1.0, 0.9])
hour_hi    = np.array([1.2, 1.2, 1.1, 1.7, 1.5, 2.0, 1.2, 1.1])

bucket = np.searchsorted(hour_edges, hour, side="right") - 1
times["hour_mult"] = rng.uniform(hour_lo[bucket], hour_hi[bucket])

# Slowdown Combined
times["multiplier"] = 0.7 * times["hour_mult"] + 0.3 * times["day_mult"]

# Merging datasets
out = base.merge(times[["started_at", "multiplier"]], how="cross")

# per-row noise so identical conditions don't give identical trips
m = out["multiplier"].to_numpy() * (1 + rng.uniform(-NOISE, NOISE, len(out)))
m = np.maximum(m, 1.0)

# We apply the slowdown only to the first our of the trip
b = out["baseline_seconds"].to_numpy()
trip = m * np.minimum(FIRST_HOUR_SEC, b) + np.maximum(0, b - FIRST_HOUR_SEC)

out["duration_seconds"] = np.round(trip).astype(int)
out["ended_at"] = out["started_at"] + pd.to_timedelta(out["duration_seconds"], unit="s")
out["distance_km"] = out["baseline_meters"]/1000.0


trips = out[[
    "origin_zone_id", "destination_zone_id", "started_at", "ended_at",
    "duration_seconds", "distance_km",
]].reset_index(drop=True)


trips = trips.sort_values('started_at')
trips = trips.reset_index(drop=True)

trips['trip_id'] = trips.index
trips['trip_id'] = 'syn_'+trips['trip_id'].astype(str)
trips['average_speed'] = None
trips['max_speed'] = None
trips['stops_count'] = None
trips['stopped_seconds'] = None
trips['data_source'] = 'trips'


We can then assign a synthetic vehicle id, making sure that each one is "free" to make the drive to keep the tip_features dataset coherent

In [7]:
import pandas as pd

ids = range(1800)
free_at = [pd.Timestamp.min] * len(ids)   

trips = trips.sort_values('started_at').reset_index(drop=True)
assigned = []

for row in trips.itertuples():
    for i, vid in enumerate(ids):
        if free_at[i] <= row.started_at:      
            assigned.append('syn_'+str(vid))
            free_at[i] = row.ended_at
            break
    else:                                      
        assigned.append(None)                  

trips['vehicle_id'] = assigned

We save the needed columns for the trip_features dataset before continuing to transform the dataset

In [9]:
trip_features = trips[['vehicle_id','trip_id','started_at', 'distance_km','origin_zone_id','destination_zone_id','duration_seconds']]

In [10]:
# Converting the datetimes to UTC with the required ISO
for col in ["started_at", "ended_at"]:
    trips[col] = (
        trips[col]
        .dt.tz_localize("America/Cancun")
        .dt.tz_convert("UTC")
        .dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    )

In [11]:
trips = trips.loc[:,['vehicle_id','origin_zone_id','destination_zone_id','started_at','ended_at','duration_seconds','distance_km','average_speed','max_speed','stops_count','stopped_seconds','data_source']]

trips.to_csv("synthetic_trips.csv", index=False) 

In [12]:
display(trips)

,vehicle_id,origin_zone_id,destination_zone_id,started_at,ended_at,duration_seconds,distance_km,average_speed,max_speed,stops_count,stopped_seconds,data_source
0,syn_0,5,16,2026-10-05T05:00:00Z,2026-10-05T08:51:03Z,13863,313.402,None,None,None,None,trips
1,syn_1,16,8,2026-10-05T05:00:00Z,2026-10-05T05:14:12Z,852,4.458,None,None,None,None,trips
2,syn_2,16,11,2026-10-05T05:00:00Z,2026-10-05T08:43:24Z,13404,295.608,None,None,None,None,trips
3,syn_3,16,20,2026-10-05T05:00:00Z,2026-10-05T05:53:39Z,3219,65.349,None,None,None,None,trips
4,syn_4,16,21,2026-10-05T05:00:00Z,2026-10-05T05:41:10Z,2470,44.873,None,None,None,None,trips
...,...,...,...,...,...,...,...,...,...,...,...,...
436795,syn_1362,13,19,2026-11-02T04:00:00Z,2026-11-02T04:27:49Z,1669,30.877,None,None,None,None,trips
436796,syn_1363,5,16,2026-11-02T04:00:00Z,2026-11-02T07:50:08Z,13808,313.402,None,None,None,None,trips
436797,syn_1364,13,11,2026-11-02T04:00:00Z,2026-11-02T07:47:20Z,13640,309.443,None,None,None,None,trips
436798,syn_1366,14,20,2026-11-02T04:00:00Z,2026-11-02T05:03:13Z,3793,76.386,None,None,None,None,trips


## Trip Feature dataset creation

In [14]:
trip_features = trip_features.sort_values('started_at')

trip_features['hour_of_day'] = trip_features['started_at'].dt.hour.to_numpy()
trip_features['day_of_week'] = trip_features['started_at'].dt.dayofweek.to_numpy()
trip_features['previous_trip_duration'] = (
    trip_features.groupby('vehicle_id')['duration_seconds'].shift(1)
)
trip_features['historical_corridor_mean_duration'] = (
    trip_features
    .groupby(['origin_zone_id', 'destination_zone_id'])['duration_seconds']
    .transform(lambda s: s.shift(1).expanding().median())
)

In [15]:
trip_features = trip_features[['trip_id','hour_of_day', 'day_of_week','distance_km','origin_zone_id','destination_zone_id','previous_trip_duration','historical_corridor_mean_duration']]
trip_features.to_csv("synthetic_trip_features.csv", index=False) 

In [16]:
trip_features

,trip_id,hour_of_day,day_of_week,distance_km,origin_zone_id,destination_zone_id,previous_trip_duration,historical_corridor_mean_duration
0,syn_0,0,0,313.402,5,16,NaN,NaN
429,syn_1,0,0,54.138,1,10,NaN,NaN
430,syn_2,0,0,183.707,1,6,NaN,NaN
431,syn_3,0,0,285.206,1,5,NaN,NaN
432,syn_4,0,0,65.366,8,20,NaN,NaN
...,...,...,...,...,...,...,...,...
436368,syn_436475,23,6,37.515,26,20,15908.0,2150.0
436369,syn_436790,23,6,71.464,8,18,12614.0,4727.0
436370,syn_436780,23,6,76.692,8,15,1621.0,5054.0
436352,syn_436760,23,6,171.977,7,22,1143.0,10413.0
